# Run the book on Google Colab

Before the first run: upload `neurodesk_upload.zip` to the root of **My Drive**.

Runtime → Change runtime type → **GPU**. Then **Runtime → Run all**.

The book installs its own tools (Chapter 01) and saves every result to Drive as soon as it is
computed. If the runtime disconnects, run all again: finished steps are skipped.

Kept on Google Drive, in `My Drive/lfbook/`:

| | Content |
|---|---|
| `results/` | everything the book computes |
| `build.log` | full jupyter-book output of the last run |
| `records/<date>_<commit>/` | a fixed copy of each successful run: executed notebooks with their outputs + the HTML book |

## 1. Google Drive

In [ ]:
import shutil, subprocess
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
SAVE  = DRIVE / 'lfbook'
BOOK  = Path('/content/new-jupyterbook')
assert (DRIVE / 'neurodesk_upload.zip').exists(), f'upload neurodesk_upload.zip to {DRIVE}'
SAVE.mkdir(exist_ok=True)

## 2. jupyter-book and the book code

In [ ]:
!pip -q install "jupyter-book<2"     # v2 drops _config.yml / _toc.yml
!rm -rf {BOOK} && git clone -q https://github.com/Zane-133/new-jupyterbook.git {BOOK}
!git -C {BOOK} log -1 --format='book commit %h  %ad  %s'

## 3. Run the whole book

Chapters run in order, 01 to 07. Hours on the first run. Only progress lines are shown here;
the full output is in `My Drive/lfbook/build.log`.

In [ ]:
!rm -rf {BOOK}/_build
!cd {BOOK} && jupyter-book build . 2>&1 | tee {SAVE}/build.log | grep --line-buffered -E "Executing|Executed|WARNING|ERROR|build succeeded|build finished"

## 4. Check for errors and keep a record

jupyter-book does not stop when a chapter fails, so the error reports are checked here. A record
is kept only when every chapter ran without error.

In [ ]:
reports = sorted((BOOK / '_build/html/reports').glob('*.err.log'))
if reports:
    for r in reports:
        print(f'\n===== {r.name} (last lines) =====')
        print(''.join(r.read_text().splitlines(keepends=True)[-25:]))
    raise RuntimeError(f'{len(reports)} chapter(s) failed; see above')

commit = subprocess.run(['git', '-C', str(BOOK), 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
record = SAVE / 'records' / f"{datetime.now():%Y%m%d-%H%M}_{commit}"
for sub in ['jupyter_execute', 'html']:                # executed notebooks, HTML book
    shutil.copytree(BOOK / '_build' / sub, record / sub)
print(f'all chapters ran; record kept in {record}')